[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C20_Frontier_Architectures_Course/01_rope/01_rope.ipynb)

# 01 · RoPE 旋转位置编码（从零实现）

目标：用 numpy 把 RoPE 完整写出来，并**数值验证**它的核心性质——内积只依赖相对距离。

路线：正弦基线 → 2D 旋转 → 高维 RoPE（rotate_half）→ 验证相对位置不变性 → 频率/波长 → 远程衰减 → ✏️ 练习 → 🧪 真实模型胶囊。

## 1 · 基线：绝对正弦位置编码

先实现原版 Transformer 的正弦编码作为对照。它把位置向量**加**到 embedding 上——相对关系要模型自己学。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def sinusoidal_pe(seq_len, d):
    pos = np.arange(seq_len)[:, None]                 # (T,1)
    i = np.arange(0, d, 2)[None, :]                   # (1,d/2)
    div = np.exp(-(np.log(10000.0)) * i / d)          # 1/10000^(2i/d)
    pe = np.zeros((seq_len, d))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div)
    return pe

PE = sinusoidal_pe(32, 16)
print('正弦 PE 形状', PE.shape, '| 取值范围 [%.2f, %.2f]' % (PE.min(), PE.max()))
assert PE.shape == (32, 16) and PE.min() >= -1.001 and PE.max() <= 1.001

## 2 · RoPE 的频率与 cos/sin 表

频率 $\theta_i=\text{base}^{-2i/d}$。用 GPT-NeoX/Llama 的 **rotate_half** 约定：把 cos/sin 复制成全维 `(T, d)`。

In [ ]:
def rope_cos_sin(seq_len, d, base=10000.0):
    assert d % 2 == 0
    j = np.arange(0, d // 2)
    inv_freq = base ** (-2.0 * j / d)        # theta_j, 长度 d/2
    pos = np.arange(seq_len)
    ang = np.outer(pos, inv_freq)            # (T, d/2)
    cos = np.concatenate([np.cos(ang), np.cos(ang)], axis=-1)  # (T, d)
    sin = np.concatenate([np.sin(ang), np.sin(ang)], axis=-1)
    return cos, sin

cos, sin = rope_cos_sin(8, 16)
print('cos/sin 形状', cos.shape, sin.shape)
print('位置 0 处 cos 应全为 1：', np.allclose(cos[0], 1.0))
assert np.allclose(cos[0], 1.0) and np.allclose(sin[0], 0.0)

## 3 · rotate_half 与 apply_rope

`apply_rope(x) = x⊙cos + rotate_half(x)⊙sin`，等价于对每一对维度做 2D 旋转。

In [ ]:
def rotate_half(x):
    d = x.shape[-1]
    x1, x2 = x[..., :d // 2], x[..., d // 2:]
    return np.concatenate([-x2, x1], axis=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

# 玩具检查：位置 0 不旋转；旋转保模长
x = rng.standard_normal((8, 16))
x0 = apply_rope(x, cos, sin)
print('位置 0 处应与原向量相同：', np.allclose(x0[0], x[0]))
print('旋转应保模长：', np.allclose(np.linalg.norm(x0, axis=-1), np.linalg.norm(x, axis=-1)))
assert np.allclose(x0[0], x[0])
assert np.allclose(np.linalg.norm(x0, axis=-1), np.linalg.norm(x, axis=-1))

## 4 · 核心验证：内积只依赖相对距离

把 query 放在位置 $m$、key 放在位置 $n$，**同时平移** $s$ 后内积应**不变**——因为它只依赖 $m-n$。

In [ ]:
T, d = 64, 16
cos, sin = rope_cos_sin(T, d)
q = rng.standard_normal(d)
k = rng.standard_normal(d)

def dot_at(m, n):
    qm = apply_rope(q[None, :], cos[m:m+1], sin[m:m+1])[0]
    kn = apply_rope(k[None, :], cos[n:n+1], sin[n:n+1])[0]
    return qm @ kn

# 相对距离相同（差=2），绝对位置不同：内积应相等
print('dot(5,3) =', round(dot_at(5, 3), 6))
print('dot(7,5) =', round(dot_at(7, 5), 6))
print('dot(20,18)=', round(dot_at(20, 18), 6))
assert np.allclose([dot_at(5,3), dot_at(7,5), dot_at(20,18)], dot_at(5,3))
print('✅ 三者几乎相等 —— 绝对位置在内积里抵消，只留相对距离')

如上：相对距离都是 2 的三组，内积一致。这就是 RoPE 的全部意义——**绝对位置在内积里抵消，只留相对距离**。

## 5 · 频率谱与波长

不同维度对的波长 $2\pi/\theta_i$ 从几个 token 到上万 token，构成多尺度时钟。

In [ ]:
def wavelengths(d, base=10000.0):
    j = np.arange(0, d // 2)
    inv = base ** (-2.0 * j / d)
    return 2 * np.pi / inv

wl = wavelengths(d)
for jj in range(d // 2):
    print(f'维度对 {jj:2d}: 波长 {wl[jj]:10.1f} token')
print('\n最短波长（高频，分辨近距离）≈', round(wl.min(), 1))
print('最长波长（低频，覆盖远距离）≈', round(wl.max(), 1))
assert wl.max() > wl.min()

## 6 · 远程衰减：RoPE 自带的距离先验

对随机 q、k，内积期望随相对距离震荡衰减。蒙特卡洛画一条曲线。

In [ ]:
def expected_dot_by_distance(d=64, base=10000.0, n_samples=2000, max_dist=128):
    cos, sin = rope_cos_sin(max_dist + 1, d, base)
    qs = rng.standard_normal((n_samples, d))
    ks = rng.standard_normal((n_samples, d))
    out = []
    for dist in range(max_dist + 1):
        qm = apply_rope(qs, cos[dist:dist+1], sin[dist:dist+1])   # query at pos=dist
        kn = apply_rope(ks, cos[0:1], sin[0:1])                   # key at pos=0
        out.append(np.mean(np.sum(qm * kn, axis=1)))
    return np.array(out)

curve = expected_dot_by_distance()
print('距离 0 处期望内积 ≈', round(curve[0], 3))
print('距离 64 处 ≈', round(curve[64], 3), ' 距离 128 处 ≈', round(curve[128], 3))
assert abs(curve[100:]).mean() < abs(curve[:20]).mean()
print('✅ 整体趋势：随距离增大幅度衰减')

> 若装了 matplotlib，可 `import matplotlib.pyplot as plt; plt.plot(curve)` 直观看到震荡衰减。

---
## ✏️ 练习 1：从零写交错式旋转 `apply_rope_interleaved`

不走 rotate_half，直接对每一对维度 $(x_{2i}, x_{2i+1})$ 做 2D 旋转。

要求：`apply_rope_interleaved(x, pos, base=10000.)`，`x` 形状 `(d,)`，对第 $i$ 对用角度 `pos*theta_i` 旋转。

In [ ]:
def apply_rope_interleaved(x, pos, base=10000.0):
    d = x.shape[-1]
    out = x.copy().astype(float)
    # TODO: 对每一对 (2i, 2i+1) 用角度 pos*base**(-2i/d) 做 2D 旋转
    # 提示：x1' = x1*cos - x2*sin ; x2' = x1*sin + x2*cos
    raise NotImplementedError
    return out

In [ ]:
# —— 练习 1 自测 ——
d8 = 8
xx = rng.standard_normal(d8)
assert np.allclose(apply_rope_interleaved(xx, 0), xx), '位置0应不变'
for pos in [1, 3, 7]:
    assert np.allclose(np.linalg.norm(apply_rope_interleaved(xx, pos)), np.linalg.norm(xx)), '旋转应保范数'
def dot_il(qq, kk, m, n):
    return apply_rope_interleaved(qq, m) @ apply_rope_interleaved(kk, n)
q2, k2 = rng.standard_normal(d8), rng.standard_normal(d8)
assert np.isclose(dot_il(q2,k2,5,3), dot_il(q2,k2,9,7)), '内积应只依赖相对距离'
print('✅ 练习 1 通过')

## ✏️ 练习 2：位置插值（Position Interpolation）

把长度 `L_test` 的位置压回训练长度 `L_train`：用缩放因子 `L_train/L_test` 乘到位置上。实现 `rope_cos_sin_pi`，验证位置 `L_test-1` 处的最大角度 ≈ 原始 `L_train-1` 处的最大角度。

In [ ]:
def rope_cos_sin_pi(seq_len, d, L_train, L_test, base=10000.0):
    # TODO: 与 rope_cos_sin 相同，但把 pos 乘以缩放因子 scale = L_train / L_test
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
d16 = 16
cos_pi, sin_pi = rope_cos_sin_pi(2048, d16, L_train=512, L_test=2048, base=10000.0)
assert cos_pi.shape == (2048, d16)
cos_ref, _ = rope_cos_sin(512, d16)
assert np.allclose(cos_pi[2044], cos_ref[511], atol=0.05), '压缩后最远相位应落回训练区间'
print('✅ 练习 2 通过（位置插值把超长序列的相位压回训练区间）')

## ✏️ 练习 3：NTK-aware base 缩放

不压位置，而是放大 base：`base' = base * scale^(d/(d-2))`，其中 `scale=L_test/L_train`。实现并验证：放大 base 后，高频（小 i）几乎不变、低频（大 i）波长显著变长。

In [ ]:
def ntk_base(base, d, L_train, L_test):
    # TODO: 返回 NTK-aware 放大后的 base'
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
d64 = 64
b0 = 10000.0
b1 = ntk_base(b0, d64, 4096, 32768)
assert b1 > b0, 'NTK base 应被放大'
wl0, wl1 = wavelengths(d64, b0), wavelengths(d64, b1)
assert np.isclose(wl1[0], wl0[0], rtol=0.05), '最高频波长几乎不变'
assert wl1[-1] > 1.5 * wl0[-1], '最低频波长应显著变长'
print('✅ 练习 3 通过 | base %.0f -> %.0f' % (b0, b1))

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def apply_rope_interleaved(x, pos, base=10000.0):
    d = x.shape[-1]
    out = x.copy().astype(float)
    for i in range(d // 2):
        theta = base ** (-2.0 * i / d)
        ang = pos * theta
        cc, ss = np.cos(ang), np.sin(ang)
        a, b = x[2*i], x[2*i+1]
        out[2*i]   = a * cc - b * ss
        out[2*i+1] = a * ss + b * cc
    return out

In [ ]:
# 练习 2 参考答案
def rope_cos_sin_pi(seq_len, d, L_train, L_test, base=10000.0):
    scale = L_train / L_test
    j = np.arange(0, d // 2)
    inv_freq = base ** (-2.0 * j / d)
    pos = np.arange(seq_len) * scale
    ang = np.outer(pos, inv_freq)
    cos = np.concatenate([np.cos(ang), np.cos(ang)], axis=-1)
    sin = np.concatenate([np.sin(ang), np.sin(ang)], axis=-1)
    return cos, sin

In [ ]:
# 练习 3 参考答案
def ntk_base(base, d, L_train, L_test):
    scale = L_test / L_train
    return base * scale ** (d / (d - 2))

---
## 🧪 真实数据胶囊：核对真实模型的 RoPE 配置

用真实开源模型的 RoPE 超参（来自其 `config.json`）算波长表，体会**长上下文模型是怎么调 RoPE 的**。这里用内置的真实数值（取自公开 config）；若需联网可自行用 transformers 拉取对照。

In [ ]:
# 真实模型 RoPE 配置（取自公开 config.json）
REAL = {
    'Llama-2-7B':   dict(dim=128, base=10000.0,  ctx=4096),
    'Llama-3-8B':   dict(dim=128, base=500000.0, ctx=8192),
    'DeepSeek-V2':  dict(dim=64,  base=10000.0,  ctx=4096),  # MLA 解耦 RoPE 维度
}

def max_wavelength(dim, base):
    j = np.arange(0, dim // 2)
    inv = base ** (-2.0 * j / dim)
    return (2 * np.pi / inv).max()

for name, cfg in REAL.items():
    wlm = max_wavelength(cfg['dim'], cfg['base'])
    print(f"{name:12s} dim={cfg['dim']:3d} base={cfg['base']:>8.0f} ctx={cfg['ctx']:>5d} | 最长波长 {wlm:>12.0f} token")

ratio = max_wavelength(128, 500000.0) / max_wavelength(128, 10000.0)
print('\nLlama-3 把 base 从 1e4 提到 5e5，最长波长放大约 %.0fx —— 支持更长上下文的关键改动之一。' % ratio)
assert ratio > 10

**🧪 胶囊练习**：实现 `min_base_for_context(dim, ctx)`：给定上下文长度，返回使**最长波长 ≥ ctx** 的最小 base（保证最低频在窗口内不卷绕）。

In [ ]:
def min_base_for_context(dim, ctx):
    # 最长波长对应 i=dim/2-1: wl = 2*pi*base**((dim-2)/dim). 解 wl>=ctx
    # TODO: 返回满足 2*pi*base**((dim-2)/dim) >= ctx 的最小 base
    raise NotImplementedError

In [ ]:
# 自测
b = min_base_for_context(128, 8192)
assert max_wavelength(128, b) >= 8192 * 0.999, '最长波长应覆盖上下文'
assert max_wavelength(128, b * 0.5) < 8192, '再小的 base 不够'
print('✅ 胶囊练习通过 | dim=128 ctx=8192 需要 base >=', round(b, 1))

In [ ]:
# 📖 胶囊参考答案
def min_base_for_context(dim, ctx):
    # 2*pi*base**((dim-2)/dim) = ctx  ->  base = (ctx/(2*pi))**(dim/(dim-2))
    return (ctx / (2 * np.pi)) ** (dim / (dim - 2))

---
### 小结
- RoPE = 把 q/k 按位置旋转，内积只依赖相对距离（复数视角一行证明）。
- 多频率 = 多尺度时钟；base 控制最长波长，是长上下文的关键旋钮。
- 外推靠 PI（压位置）或 NTK（放大 base），C25 会深入 YaRN。

下一站：**模块 02 · 注意力变体 MHA→GQA→MLA**。